In [ ]:
import random
import math
import json
import os
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Polygon, Rectangle

# -------------------------
# configuration
# -------------------------

COLORS = {
    "yellow": "#FFD700",
    "blue": "#0099ff",
    "black": "#000000",
    "red": "#ff3333",
    "green": "#33cc33"
}

SHAPES = ["circle", "triangle", "square"]
SIZES = [8, 12, 16]

CANVAS_SIZE = 100
MIN_DIST = 15

OUT_DIR = "reference_game_dataset"
os.makedirs(OUT_DIR, exist_ok=True)


# -------------------------
# utilities
# -------------------------

def distance(a, b):
    return math.sqrt((a["x_loc"] - b["x_loc"])**2 + (a["y_loc"] - b["y_loc"])**2)


def sample_position(existing):

    while True:

        x = random.randint(10, 90)
        y = random.randint(10, 90)

        candidate = {"x_loc": x, "y_loc": y}

        if all(distance(candidate, obj) > MIN_DIST for obj in existing):
            return x, y


def generate_object(existing, color=None, shape=None, size=None):

    if color is None:
        color = random.choice(list(COLORS.keys()))

    if shape is None:
        shape = random.choice(SHAPES)

    if size is None:
        size = random.choice(SIZES)

    x, y = sample_position(existing)

    return {
        "x_loc": x,
        "y_loc": y,
        "color": color,
        "type": shape,
        "size": size
    }


# -------------------------
# pragmatic scene generator
# -------------------------

def generate_scene(n_objects=6):

    objects = []

    # target
    target = generate_object(objects)
    objects.append(target)

    # controlled distractors
    dims = ["color", "type", "size"]

    for dim in dims:

        obj = target.copy()

        if dim == "color":
            obj["color"] = random.choice([c for c in COLORS if c != target["color"]])

        if dim == "type":
            obj["type"] = random.choice([s for s in SHAPES if s != target["type"]])

        if dim == "size":
            obj["size"] = random.choice([s for s in SIZES if s != target["size"]])

        x, y = sample_position(objects)
        obj["x_loc"] = x
        obj["y_loc"] = y

        objects.append(obj)

    while len(objects) < n_objects:
        objects.append(generate_object(objects))

    return objects


# -------------------------
# drawing shapes
# -------------------------

def draw_shape(ax, obj):

    x = obj["x_loc"]
    y = obj["y_loc"]
    size = obj["size"]
    color = COLORS[obj["color"]]

    if obj["type"] == "circle":

        patch = Circle((x, y), size, color=color)

    elif obj["type"] == "square":

        patch = Rectangle((x-size, y-size), size*2, size*2, color=color)

    elif obj["type"] == "triangle":

        triangle = [
            (x, y + size),
            (x - size, y - size),
            (x + size, y - size)
        ]

        patch = Polygon(triangle, color=color)

    ax.add_patch(patch)


# -------------------------
# render scene
# -------------------------

def render_scene(scene, filename):

    fig, ax = plt.subplots(figsize=(4,4))

    ax.set_xlim(0, CANVAS_SIZE)
    ax.set_ylim(0, CANVAS_SIZE)

    ax.set_xticks([])
    ax.set_yticks([])

    for obj in scene:
        draw_shape(ax, obj)

    plt.savefig(filename, bbox_inches="tight")
    plt.close()


# -------------------------
# dataset generation
# -------------------------

def generate_dataset(n=50):

    dataset = []

    for i in range(n):

        scene = generate_scene()

        img_path = f"{OUT_DIR}/scene_{i}.png"

        render_scene(scene, img_path)

        dataset.append({
            "image": img_path,
            "objects": scene
        })

    with open(f"{OUT_DIR}/dataset.json", "w") as f:
        json.dump(dataset, f, indent=2)


generate_dataset(100)